# P·π 분포 분석 및 Q 전략 설계

**목적**: BL 모델에서 Q 설정 전에 P·π의 실제 분포를 확인하고,  
레짐별로 P·π가 어떻게 달라지는지 분석하여 Q 전략의 데이터 기반 근거를 마련.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from pathlib import Path
from scipy import stats as sp_stats

warnings.filterwarnings('ignore')

from bl_functions import (
    compute_sigma, compute_daily_slice, compute_pi, build_P,
)

BASE_DIR  = Path.cwd()
DATA_DIR  = BASE_DIR / 'data'
OUT_DIR   = BASE_DIR / 'results'
OUT_DIR.mkdir(exist_ok=True)

# HMM 결과 경로 (정제 n=5 사용)
HMM_DIR  = Path(r'../서윤범/low_risk/outputs')
HMM_FILE = HMM_DIR / 'hmm_refined_n5_results.csv'

# BL 공통 파라미터
TRAIN_WINDOW = 60
THRESH_DAILY = 0.9
PCT_GROUP    = 0.30
START_PRED   = '2010-01-01'
LAM_FIXED    = 2.5

print('설정 완료')

설정 완료


In [2]:
# ── 데이터 로드 ────────────────────────────────────────────────
panel     = pd.read_csv(DATA_DIR / 'monthly_panel.csv', parse_dates=['date'])
panel     = panel.set_index(['date', 'ticker'])
daily_ret = pd.read_pickle(DATA_DIR / 'daily_returns.pkl')

all_dates  = panel.index.get_level_values('date').unique().sort_values()
pred_dates = all_dates[all_dates >= START_PRED]

spy_series = panel['spy_ret'].groupby(level='date').first()
rf_series  = panel['rf_1m'].groupby(level='date').first()

# ── HMM 레짐 로드 ──────────────────────────────────────────────
hmm_df = pd.read_csv(HMM_FILE, parse_dates=['date'], index_col='date')
print(f'Panel:    {panel.shape}')
print(f'pred_dates: {len(pred_dates)}개  ({pred_dates[0].date()} ~ {pred_dates[-1].date()})')
print(f'HMM:      {hmm_df.shape},  레짐: {hmm_df["regime"].unique().tolist()}')

Panel:    (97944, 11)
pred_dates: 180개  (2010-01-31 ~ 2024-12-31)
HMM:      (5512, 10),  레짐: ['Mild Bull (완만)', 'Neutral (중립)', 'Bull (강세)', 'Volatile (고변동)', 'Crisis (위기)']


In [3]:
# ══════════════════════════════════════════════════════════════
# P·π 시계열 계산
# 매 리밸런싱 시점에서 Sigma, w_mkt, P를 구성하고 P·π 스칼라 계산
# ══════════════════════════════════════════════════════════════
records = []

for pred_date in pred_dates:
    try:
        month_df = panel.xs(pred_date, level='date').dropna(
            subset=['vol_21d', 'log_mcap', 'ret_1m'])
        if len(month_df) < 30:
            continue

        daily_slice, valid_tix = compute_daily_slice(
            pred_date, month_df.index.tolist(),
            daily_ret, TRAIN_WINDOW, THRESH_DAILY)
        if len(valid_tix) < 20:
            continue

        Sigma    = compute_sigma(daily_slice, scale=21)
        month_df = month_df.reindex(valid_tix)
        mcap     = np.exp(month_df['log_mcap'])
        w_mkt    = mcap / mcap.sum()

        # 시장 분산 계산
        idx         = all_dates.get_loc(pred_date)
        train_dates = all_dates[max(0, idx - TRAIN_WINDOW): idx]
        spy_s       = spy_series.reindex(train_dates)
        rf_s        = rf_series.reindex(train_dates)
        spy_excess  = float((spy_s - rf_s).mean() * 21)
        sigma2_mkt  = float(spy_s.std() ** 2 * 21)

        pi, lam = compute_pi(Sigma, w_mkt, spy_excess, sigma2_mkt, lam_fixed=LAM_FIXED)

        # P 구성 (mcap 가중)
        P = build_P(month_df['vol_21d'], mcap, pct=PCT_GROUP, weighting='mcap')

        p_pi = float(P @ pi)   # 핵심 스칼라

        records.append({
            'date'     : pred_date,
            'p_pi'     : p_pi,
            'lam'      : lam,
            'sigma2_mkt': sigma2_mkt,
            'n_stocks' : len(valid_tix),
        })
    except Exception as e:
        pass

p_pi_df = pd.DataFrame(records).set_index('date')
print(f'계산 완료: {len(p_pi_df)}개 시점')
print(f'P·π 평균:  {p_pi_df["p_pi"].mean():.6f}')
print(f'P·π 범위:  {p_pi_df["p_pi"].min():.6f} ~ {p_pi_df["p_pi"].max():.6f}')
print(f'P·π < 0 비율: {(p_pi_df["p_pi"] < 0).mean()*100:.1f}%')

계산 완료: 180개 시점
P·π 평균:  -0.004029
P·π 범위:  -0.013284 ~ -0.000215
P·π < 0 비율: 100.0%


In [ ]:
# ── HMM 레짐과 병합 ────────────────────────────────────────────
# pred_date는 월말 → HMM은 일별이므로 월말 기준 레짐 추출
hmm_monthly = hmm_df['regime'].resample('ME').last()
hmm_monthly.index = hmm_monthly.index.to_period('M').to_timestamp('M')

p_pi_df = p_pi_df.copy()
p_pi_df.index = pd.to_datetime(p_pi_df.index)

# 가장 가까운 날짜 기준 merge
p_pi_df['regime'] = p_pi_df.index.map(
    lambda d: hmm_df['regime'].asof(d) if d >= hmm_df.index[0] else np.nan
)

p_pi_regime = p_pi_df.dropna(subset=['regime'])
print(f'레짐 매핑 완료: {len(p_pi_regime)}개 시점')
print(p_pi_regime['regime'].value_counts())

In [ ]:
# ══════════════════════════════════════════════════════════════
# 시각화 1: P·π 시계열 + 레짐 색상
# ══════════════════════════════════════════════════════════════
REGIME_COLORS = {
    'Bull (강세)'      : '#2ecc71',
    'Mild Bull (완만)' : '#27ae60',
    'Neutral (중립)'   : '#f39c12',
    'Volatile (고변동)': '#e67e22',
    'Crisis (위기)'    : '#e74c3c',
    'Bear (약세)'      : '#c0392b',
    'Recovery (회복)'  : '#3498db',
}
DEFAULT_C = '#95a5a6'

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# 상단: P·π 시계열
ax = axes[0]
ax.plot(p_pi_regime.index, p_pi_regime['p_pi'], color='#2c3e50', lw=1.0)
ax.axhline(0, color='red', lw=1.2, linestyle='--', alpha=0.7, label='P·π = 0')
ax.fill_between(p_pi_regime.index, p_pi_regime['p_pi'], 0,
                where=(p_pi_regime['p_pi'] < 0), alpha=0.2, color='#e74c3c',
                label='P·π < 0 (CAPM이 저변동 불리)')
ax.fill_between(p_pi_regime.index, p_pi_regime['p_pi'], 0,
                where=(p_pi_regime['p_pi'] > 0), alpha=0.2, color='#2ecc71',
                label='P·π > 0 (CAPM이 저변동 유리)')
ax.set_ylabel('P·π (월간 수익률)')
ax.set_title('P·π 시계열 — CAPM 균형에서 저변동 프리미엄 위치', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 하단: 레짐별 색상 배경
ax2 = axes[1]
regimes = p_pi_regime['regime']
prev_r  = regimes.iloc[0]; t0 = p_pi_regime.index[0]
for t, r in zip(p_pi_regime.index[1:], regimes.iloc[1:]):
    if r != prev_r:
        ax2.axvspan(t0, t, alpha=0.4,
                    color=REGIME_COLORS.get(prev_r, DEFAULT_C), lw=0)
        prev_r = r; t0 = t
ax2.axvspan(t0, p_pi_regime.index[-1], alpha=0.4,
            color=REGIME_COLORS.get(prev_r, DEFAULT_C), lw=0)
ax2.plot(p_pi_regime.index, p_pi_regime['p_pi'], color='#2c3e50', lw=1.0)
ax2.axhline(0, color='red', lw=1.0, linestyle='--', alpha=0.7)
ax2.set_ylabel('P·π'); ax2.set_xlabel('날짜')
ax2.set_title('레짐별 P·π (배경색 = 레짐)', fontweight='bold')
patches = [mpatches.Patch(color=v, label=k, alpha=0.6)
           for k, v in REGIME_COLORS.items() if k in regimes.unique()]
ax2.legend(handles=patches, fontsize=8, ncol=3); ax2.grid(alpha=0.3)

plt.suptitle('P·π 시계열 분석', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'ppi_timeseries.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# 시각화 2: 레짐별 P·π 분포 비교
# ══════════════════════════════════════════════════════════════
regime_order = sorted(
    p_pi_regime['regime'].unique(),
    key=lambda r: p_pi_regime[p_pi_regime['regime']==r]['p_pi'].mean()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 박스플롯
box_data = [p_pi_regime[p_pi_regime['regime']==r]['p_pi'].values * 100
            for r in regime_order]
colors_b = [REGIME_COLORS.get(r, DEFAULT_C) for r in regime_order]
bp = axes[0].boxplot(box_data, labels=regime_order, patch_artist=True,
                     medianprops=dict(color='black', lw=2))
for patch, c in zip(bp['boxes'], colors_b):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0].axhline(0, color='red', lw=1.2, linestyle='--', alpha=0.8, label='P·π=0')
axes[0].set_title('레짐별 P·π 분포', fontweight='bold', fontsize=12)
axes[0].set_ylabel('P·π × 100 (bp)'); axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# KDE
for r in regime_order:
    vals = p_pi_regime[p_pi_regime['regime']==r]['p_pi'] * 100
    vals.plot.kde(ax=axes[1], label=r, color=REGIME_COLORS.get(r, DEFAULT_C), lw=2)
axes[1].axvline(0, color='red', lw=1.2, linestyle='--', alpha=0.8, label='P·π=0')
axes[1].set_title('레짐별 P·π 밀도 분포', fontweight='bold', fontsize=12)
axes[1].set_xlabel('P·π × 100 (bp)'); axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('레짐별 P·π 분포 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'ppi_regime_dist.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# 수치 요약 + Kruskal-Wallis 검정
# ══════════════════════════════════════════════════════════════
print('=== 레짐별 P·π 통계 (단위: bp = ×100) ===\n')
print(f'{"레짐":<20} {"평균":>8} {"중앙값":>8} {"std":>8} {"<0 비율":>8} {"건수":>6}')
print('-' * 62)

groups = []
for r in regime_order:
    vals = p_pi_regime[p_pi_regime['regime']==r]['p_pi'] * 100
    groups.append(vals.values)
    neg_pct = (vals < 0).mean() * 100
    print(f'  {r:<18} {vals.mean():>+7.3f}  {vals.median():>+7.3f}  '
          f'{vals.std():>7.3f}  {neg_pct:>7.1f}%  {len(vals):>5}')

h_stat, p_val = sp_stats.kruskal(*groups)
sig = '유의미하게 다름 ✅' if p_val < 0.05 else '유의하지 않음 ❌'
print(f'\nKruskal-Wallis  H={h_stat:.2f},  p={p_val:.4f}  ({sig})')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Q 전략 함의: Q - P·π 분포
# 현재 Q 고정값에서 실제 view surprise가 얼마나 되는지 확인
# ══════════════════════════════════════════════════════════════
Q_FIXED = 0.003  # 현재 사용 중인 고정 Q값 (월간)

p_pi_regime['view_surprise'] = Q_FIXED - p_pi_regime['p_pi']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# view surprise 시계열
axes[0].plot(p_pi_regime.index, p_pi_regime['view_surprise'] * 100,
             color='#8e44ad', lw=1.0)
axes[0].axhline(Q_FIXED * 100, color='gray', lw=1.0, linestyle='--',
                alpha=0.8, label=f'Q={Q_FIXED*100:.2f}bp (P·π=0 기준)')
axes[0].set_title(f'Q - P·π 시계열 (view surprise, Q={Q_FIXED*100:.2f}bp 고정)',
                  fontweight='bold', fontsize=11)
axes[0].set_ylabel('Q - P·π (bp)'); axes[0].legend(); axes[0].grid(alpha=0.3)

# 레짐별 view surprise 평균
vs_means = p_pi_regime.groupby('regime')['view_surprise'].mean() * 100
vs_means = vs_means.reindex(regime_order)
bar_colors = [REGIME_COLORS.get(r, DEFAULT_C) for r in vs_means.index]
axes[1].bar(vs_means.index, vs_means.values, color=bar_colors, edgecolor='white', alpha=0.85)
axes[1].axhline(Q_FIXED * 100, color='gray', lw=1.2, linestyle='--',
                alpha=0.8, label=f'Q={Q_FIXED*100:.2f}bp')
axes[1].set_title('레짐별 평균 Q - P·π (view surprise)', fontweight='bold', fontsize=11)
axes[1].set_ylabel('평균 Q - P·π (bp)'); axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
for i, (r, v) in enumerate(vs_means.items()):
    axes[1].text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle('Q - P·π (view surprise) 분석', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'ppi_view_surprise.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n=== 레짐별 평균 view surprise ===')
print('(Q - P·π: 클수록 CAPM 균형 대비 강한 저변동 뷰)')
print(vs_means.to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════
# Q 전략 설계 제안
# P·π 분포를 기반으로 레짐별 Q 값 계산
# ══════════════════════════════════════════════════════════════
TARGET_SURPRISE = Q_FIXED - p_pi_regime['p_pi'].mean()  # 전체 평균 기준 view surprise

print('=== Q 전략 설계 제안 ===\n')
print(f'고정 Q = {Q_FIXED*100:.3f}bp 유지 시 레짐별 실제 view surprise:')
print(f'  전체 평균 P·π = {p_pi_regime["p_pi"].mean()*100:.3f}bp')
print(f'  목표 view surprise = {TARGET_SURPRISE*100:.3f}bp\n')

print(f'{"레짐":<20} {"평균 P·π":>10} {"현재 surprise":>14} {"제안 Q (surplus 유지)":>22}')
print('-' * 72)

for r in regime_order:
    mean_ppi = p_pi_regime[p_pi_regime['regime']==r]['p_pi'].mean()
    curr_surprise = (Q_FIXED - mean_ppi) * 100
    q_suggested   = (TARGET_SURPRISE + mean_ppi) * 100  # 동일 surprise 유지
    print(f'  {r:<18} {mean_ppi*100:>+9.3f}bp  {curr_surprise:>13.3f}bp  {q_suggested:>21.3f}bp')

print(f'\n해석:')
print(f'  P·π가 더 음수인 레짐 → Q 고정 시 view surprise가 자동으로 커짐')
print(f'  → 레짐별 동일한 저변동 신호 강도를 원한다면 Q를 조정해야 함')
print(f'  → P·π가 이미 충분히 음수라면 Q를 작게 설정해도 됨')